# Notebook 07 — Process Fidelity for a CZ-Inspired Pulse Sequence

This notebook upgrades the gate benchmark from state-by-state overlaps to a **true unitary comparison**.

We compare the effective operation generated by the minimal pulse sequence against the ideal controlled-Z unitary:

$$
U_{\mathrm{CZ}} = \mathrm{diag}(1, 1, 1, -1)
$$

For a unitary $U$ acting on a $d$-dimensional computational space, we use the standard trace-overlap process metric:

$$
F_{\mathrm{proc}} = \frac{|\mathrm{Tr}(U_{\mathrm{CZ}}^\dagger U)|^2}{d^2}
$$

Here, $d=4$ for the two-qubit computational basis.

This notebook focuses on the **coherent** case, where the evolution is unitary and an effective gate matrix can be built directly from evolved basis states.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, sigmay, sesolve, Qobj

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
sy = sigmay()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
sy1 = tensor(sy, I)
sy2 = tensor(I, sy)
n1 = tensor(n, I)
n2 = tensor(I, n)

# Ideal CZ unitary in the computational basis [|gg>, |gr>, |rg>, |rr>]
U_cz = np.diag([1, 1, 1, -1])


## Hamiltonian and pulse sequence

In [ ]:
def two_atom_hamiltonian(omega1: float, omega2: float, delta: float, V: float, axis: str = 'x'):
    if axis == 'x':
        drive = 0.5 * omega1 * sx1 + 0.5 * omega2 * sx2
    elif axis == 'y':
        drive = 0.5 * omega1 * sy1 + 0.5 * omega2 * sy2
    else:
        raise ValueError("axis must be 'x' or 'y'")

    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def run_segment(state, omega1: float, omega2: float, delta: float, V: float,
                duration: float, axis: str = 'x', n_steps: int = 160):
    H = two_atom_hamiltonian(omega1=omega1, omega2=omega2, delta=delta, V=V, axis=axis)
    times = np.linspace(0.0, duration, n_steps)
    result = sesolve(H, state, times)
    return result.states[-1], times, result.states


def run_minimal_sequence(psi0, omega: float, delta: float, V: float,
                         t_pi: float, t_mid: float, axis: str = 'x'):
    state1, times1, states1 = run_segment(
        psi0, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis
    )
    state2, times2, states2 = run_segment(
        state1, omega1=0.0, omega2=omega, delta=delta, V=V,
        duration=t_mid, axis=axis
    )
    state3, times3, states3 = run_segment(
        state2, omega1=omega, omega2=0.0, delta=delta, V=V,
        duration=t_pi, axis=axis
    )

    full_times = np.concatenate([
        times1,
        times2 + times1[-1],
        times3 + times1[-1] + times2[-1],
    ])
    full_states = states1 + states2 + states3
    return full_times, full_states, state3


## Build the effective gate matrix

We evolve each computational basis state through the pulse sequence and assemble the output columns into an effective matrix $U_{\mathrm{eff}}$.


In [ ]:
def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)


def build_effective_unitary(omega: float, delta: float, V: float, t_pi: float, t_mid: float):
    columns = []
    final_states = []
    for psi0 in basis_states:
        _, _, psi_final = run_minimal_sequence(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            t_pi=t_pi,
            t_mid=t_mid,
            axis='x',
        )
        columns.append(state_to_column(psi_final))
        final_states.append(psi_final)

    U_eff = np.column_stack(columns)
    return U_eff, final_states


def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))


def unitarity_error(U_eff):
    identity = np.eye(U_eff.shape[0], dtype=complex)
    return float(np.linalg.norm(np.conjugate(U_eff.T) @ U_eff - identity))


## Evaluate one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_pi = np.pi / omega
t_mid = 2 * np.pi / omega

U_eff, final_states = build_effective_unitary(omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid)
F_proc = process_fidelity(U_eff)
unit_err = unitarity_error(U_eff)

print('Effective gate matrix U_eff:')
print(np.round(U_eff, 3))
print()
print(f'Process fidelity vs ideal CZ: {F_proc:.6f}')
print(f'Unitarity error ||U^†U - I||:   {unit_err:.6e}')

## Visualize the effective gate matrix magnitude

In [ ]:
plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_eff), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for the minimal pulse sequence')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()

## Scan the middle pulse duration

In [ ]:
t_mid_vals = np.linspace(0.2 * np.pi / omega, 3.2 * np.pi / omega, 45)
proc_vals = []
unit_vals = []

for t_mid_test in t_mid_vals:
    U_eff_test, _ = build_effective_unitary(
        omega=omega,
        delta=delta,
        V=V,
        t_pi=t_pi,
        t_mid=t_mid_test,
    )
    proc_vals.append(process_fidelity(U_eff_test))
    unit_vals.append(unitarity_error(U_eff_test))

In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, proc_vals, label='process fidelity')
plt.axvline(2 * np.pi / omega, linestyle='--', label=r'$2\pi/\Omega$')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel(r'$F_{proc}$')
plt.title('Process fidelity versus middle pulse duration')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, unit_vals)
plt.axvline(2 * np.pi / omega, linestyle='--')
plt.xlabel(r'Middle pulse duration $t_{mid}$')
plt.ylabel(r'Unitarity error $||U^\dagger U - I||$')
plt.title('Unitarity error versus middle pulse duration')
plt.tight_layout()
plt.show()

## Process-fidelity landscape over $(\Omega, V)$

In [ ]:
omega_vals = np.linspace(0.4, 2.0, 24)
V_vals = np.linspace(0.0, 8.0, 24)

proc_landscape = np.zeros((len(V_vals), len(omega_vals)))
unit_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V_scan in enumerate(V_vals):
    for j, omega_scan in enumerate(omega_vals):
        t_pi_scan = np.pi / omega_scan
        t_mid_scan = 2 * np.pi / omega_scan
        U_eff_scan, _ = build_effective_unitary(
            omega=omega_scan,
            delta=0.0,
            V=V_scan,
            t_pi=t_pi_scan,
            t_mid=t_mid_scan,
        )
        proc_landscape[i, j] = process_fidelity(U_eff_scan)
        unit_landscape[i, j] = unitarity_error(U_eff_scan)

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    proc_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, proc_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Process fidelity over $(\Omega, V)$')
plt.colorbar(im, label=r'$F_{proc}$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    unit_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, unit_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Unitarity error over $(\Omega, V)$')
plt.colorbar(im, label=r'$||U^\dagger U - I||$')
plt.tight_layout()
plt.show()

## Best point in the $(\Omega, V)$ scan

In [ ]:
best_index = np.unravel_index(np.argmax(proc_landscape), proc_landscape.shape)
best_V = V_vals[best_index[0]]
best_omega = omega_vals[best_index[1]]
best_proc = proc_landscape[best_index]
best_unit = unit_landscape[best_index]

print(f'Best scan point: Omega = {best_omega:.3f}, V = {best_V:.3f}, process fidelity = {best_proc:.6f}, unitarity error = {best_unit:.6e}')

## Interpretation

This notebook converts the pulse-sequence benchmark into a true gate metric.

Key readouts are:
- whether the effective operation is close to the ideal CZ matrix,
- whether the operation remains close to unitary on the computational subspace,
- whether the best process-fidelity region matches the same blockade-dominated regime seen in earlier notebooks.

This is the cleanest capstone metric in the repo because it reports a full matrix-level comparison rather than only state-by-state overlaps.


## Optional next steps

Natural extensions are:

- include explicit pulse phases and $\sigma_y$ rotations,
- add detuning ramps during the middle pulse,
- project noisy dynamics onto a quantum channel and compare to the ideal CZ map,
- use numerical optimal control to improve process fidelity.
